# TC2 - Introdução ao Reconhecimento de Padrões (TI0097)

Trabalho sobre estimação, comparação e inversão da matriz de covariância, usando a base Wall Following Robot Navigation Data com os arquivos de 4 e de 24 sensores. Aqui todas as 4 classes originais são mantidas, já que o enunciado não pede nenhuma redução.

Os arquivos sensor_readings_4.data e sensor_readings_24.data precisam estar na mesma pasta deste notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

sns.set_style("whitegrid")
np.random.seed(42)

## Carregando os dois conjuntos de dados

Cada linha da base é uma amostra e a última coluna é o rótulo da classe. Guardando a matriz de atributos já transposta (p x N), do jeito que os slides da disciplina definem X, com os atributos nas linhas e as amostras nas colunas.

In [ ]:
def carregar(caminho):
    df = pd.read_csv(caminho, header=None)
    y = df.iloc[:, -1].values
    X = df.iloc[:, :-1].values.T.astype(float)  # p x N
    return X, y

X4, y4 = carregar("sensor_readings_4.data")
X24, y24 = carregar("sensor_readings_24.data")
classes = sorted(pd.unique(y4))
print("4 sensores: p =", X4.shape[0], ", N =", X4.shape[1])
print("24 sensores: p =", X24.shape[0], ", N =", X24.shape[1])
print("classes:", classes)

## Questão 1 - Estimação da matriz de covariância global

Implementando os 4 métodos vistos em aula. Todos recebem X no formato p x N.

Método 1 usa um laço percorrendo as N colunas e soma os produtos externos (X(:,j)-m)(X(:,j)-m)'.
Método 2 faz a mesma conta só que vetorizada, centralizando a matriz inteira de uma vez.
Método 3 também usa laço, mas acumula a matriz de correlação R e só subtrai m*m' no final.
Método 4 é a versão vetorizada do método 3, calculando R = XX'/N direto.

In [ ]:
def mcovar1(X):
    p, N = X.shape
    m = X.mean(axis=1, keepdims=True)
    soma = np.zeros((p, p))
    for j in range(N):
        aux = X[:, [j]] - m
        soma += aux @ aux.T
    C = soma / N
    return (C + C.T) / 2

def mcovar2(X):
    p, N = X.shape
    m = X.mean(axis=1, keepdims=True)
    aux = X - m
    C = aux @ aux.T / N
    return (C + C.T) / 2

def mcovar3(X):
    p, N = X.shape
    m = X.mean(axis=1, keepdims=True)
    R = np.zeros((p, p))
    for j in range(N):
        R += X[:, [j]] @ X[:, [j]].T
    C = R / N - m @ m.T
    return (C + C.T) / 2

def mcovar4(X):
    p, N = X.shape
    m = X.mean(axis=1, keepdims=True)
    R = X @ X.T / N
    C = R - m @ m.T
    return (C + C.T) / 2

metodos = {"Método 1": mcovar1, "Método 2": mcovar2, "Método 3": mcovar3, "Método 4": mcovar4}

A referência é o np.cov nativo, usando bias=True pra dividir por N em vez de N-1, já que os métodos implementados também dividem por N.

In [ ]:
for nome_base, X in [("4 sensores", X4), ("24 sensores", X24)]:
    Cref = np.cov(X, bias=True)
    print(f"--- {nome_base} ---")
    for nome, f in metodos.items():
        C = f(X)
        erro = np.linalg.norm(C - Cref, ord="fro")
        print(f"{nome}: norma de Cmy - Cref = {erro:.2e}")
    print()

A norma da diferença fica próxima de zero (erro numérico de máquina) em todos os métodos e nas duas bases, confirmando que as 4 formas de calcular chegam na mesma matriz de covariância, só mudando a forma de implementação.

## Questão 2 - Comparação do tempo de execução

Rodando cada método (mais o np.cov nativo) 100 vezes seguidas para cada base e guardando o tempo de cada rodada.

In [ ]:
n_rodadas = 100
linhas_tempo = []

for nome_base, X in [("4 sensores", X4), ("24 sensores", X24)]:
    for rodada in range(n_rodadas):
        for nome, f in metodos.items():
            t0 = time.perf_counter()
            f(X)
            tempo = time.perf_counter() - t0
            linhas_tempo.append({"base": nome_base, "metodo": nome, "rodada": rodada, "tempo": tempo})
        t0 = time.perf_counter()
        np.cov(X, bias=True)
        tempo = time.perf_counter() - t0
        linhas_tempo.append({"base": nome_base, "metodo": "cov nativo", "rodada": rodada, "tempo": tempo})

tempos = pd.DataFrame(linhas_tempo)
tempos.head()

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
for eixo, nome_base in zip(eixos, ["4 sensores", "24 sensores"]):
    subset = tempos[tempos["base"] == nome_base]
    for nome in subset["metodo"].unique():
        eixo.hist(subset[subset["metodo"] == nome]["tempo"], bins=25, alpha=0.5, label=nome)
    eixo.set_title(f"Histograma dos tempos ({nome_base})")
    eixo.set_xlabel("tempo (s)")
    eixo.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
for eixo, nome_base in zip(eixos, ["4 sensores", "24 sensores"]):
    subset = tempos[tempos["base"] == nome_base]
    sns.violinplot(data=subset, x="metodo", y="tempo", ax=eixo)
    eixo.set_title(f"Violin plot dos tempos ({nome_base})")
    eixo.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
resumo_tempo = tempos.groupby(["base", "metodo"])["tempo"].agg(["mean", "std"]).round(6)
resumo_tempo

Os métodos com laço explícito (Método 1 e Método 3) são visivelmente mais lentos, principalmente na base de 24 sensores, porque o laço em Python percorre uma a uma as N amostras. Já os métodos vetorizados (Método 2 e Método 4) e o np.cov nativo ficam bem mais rápidos, sendo o Método 4 geralmente o mais eficiente dos implementados, o que confirma o que já era esperado pela forma como cada fórmula foi derivada.

## Questão 3 - Covariância por classe e invertibilidade

Escolhendo o método mais rápido entre os 4 implementados (com base na tabela de tempos acima) para estimar a covariância de cada classe, e comparando o posto e o número de condicionamento dessas matrizes com os da matriz de covariância global.

In [ ]:
apenas_metodos = tempos[tempos["metodo"] != "cov nativo"]
media_por_metodo = apenas_metodos.groupby("metodo")["tempo"].mean()
metodo_mais_rapido = media_por_metodo.idxmin()
print("Método mais rápido escolhido:", metodo_mais_rapido)
funcao_escolhida = metodos[metodo_mais_rapido]

In [ ]:
def avaliar_invertibilidade(nome_base, X, y):
    print(f"--- {nome_base} ---")
    p = X.shape[0]
    C_global = funcao_escolhida(X)
    posto = np.linalg.matrix_rank(C_global)
    cond = np.linalg.cond(C_global)
    print(f"Global: p={p}, posto={posto}, número de condicionamento={cond:.3e}, rcond~={1/cond:.3e}")
    for c in classes:
        Xc = X[:, y == c]
        Cc = funcao_escolhida(Xc)
        posto_c = np.linalg.matrix_rank(Cc)
        cond_c = np.linalg.cond(Cc)
        print(f"Classe {c}: N={Xc.shape[1]}, posto={posto_c}, número de condicionamento={cond_c:.3e}, rcond~={1/cond_c:.3e}")
    print()
    return C_global

C_global_4 = avaliar_invertibilidade("4 sensores", X4, y4)
C_global_24 = avaliar_invertibilidade("24 sensores", X24, y24)

Quando o posto de uma matriz é igual a p (a dimensão do vetor de atributos), ela é de posto completo e portanto invertível. O número de condicionamento indica o quão bem-condicionada ela é na prática, valores muito altos (ordem de 10^10 pra cima) sinalizam que a matriz é numericamente instável mesmo sendo tecnicamente invertível, o que costuma acontecer mais na base de 24 sensores por causa da forte correlação entre sensores vizinhos.

## Questão 4 - Inversão e regularização das matrizes de covariância

Tentando inverter a matriz global e as matrizes de cada classe. Quando o posto não é completo ou o número de condicionamento é alto demais, aplicando a regularização vista em aula, que soma um múltiplo pequeno da identidade: Ci(λ) = Ci + λI.

In [ ]:
LIMIAR_CONDICIONAMENTO = 1e10
LAMBDA_REGULARIZACAO = 1e-6

def inverter_com_regularizacao(C, nome):
    p = C.shape[0]
    posto = np.linalg.matrix_rank(C)
    cond = np.linalg.cond(C)
    if posto < p or cond > LIMIAR_CONDICIONAMENTO:
        C_reg = C + LAMBDA_REGULARIZACAO * np.eye(p)
        invC = np.linalg.inv(C_reg)
        print(f"{nome}: precisou de regularização (posto={posto}/{p}, cond={cond:.2e})")
    else:
        invC = np.linalg.inv(C)
        print(f"{nome}: inversão direta, sem regularização (posto={posto}/{p}, cond={cond:.2e})")
    return invC

def inverter_tudo(nome_base, X, y):
    print(f"--- {nome_base} ---")
    C_global = funcao_escolhida(X)
    inverter_com_regularizacao(C_global, "Covariância global")
    for c in classes:
        Xc = X[:, y == c]
        Cc = funcao_escolhida(Xc)
        inverter_com_regularizacao(Cc, f"Classe {c}")
    print()

inverter_tudo("4 sensores", X4, y4)
inverter_tudo("24 sensores", X24, y24)

No fim das contas, a base de 4 sensores tende a não precisar de regularização, já que tem poucos atributos e bastante amostra por classe. Já a base de 24 sensores é mais propensa a exigir a regularização, principalmente nas classes com menos amostras, porque os 24 sensores captam distâncias parecidas entre ângulos vizinhos e isso deixa a matriz de covariância mal condicionada ou até com posto incompleto.